# Book Recommendation Engine

**Objective:** Recommend books from reader-rating patterns using item-based nearest neighbours.

The workflow uses the Book-Crossing files already committed in this folder, removes deprecated APIs, validates identifiers, and preserves a reproducible popularity threshold.


## 1. Setup and portable paths


In [1]:
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="deep")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__} | NumPy: {np.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Random seed: {RANDOM_STATE}")

from pathlib import Path

try:
    PROJECT_DIR = Path(__file__).resolve().parent
except NameError:
    PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "Books Dataset").exists():
    PROJECT_DIR = Path.cwd() / "Unsupervised Learning Projects" / "Book Recommendation Engine"
DATA_DIR = PROJECT_DIR / "Books Dataset"
print(f"Resolved dataset folder: {DATA_DIR.name}")


Python: 3.12.13
pandas: 2.2.3 | NumPy: 2.3.5 | scikit-learn: 1.8.0
Random seed: 42
Resolved dataset folder: Books Dataset


## 2. Load and validate Book-Crossing data


In [2]:
books = pd.read_csv(DATA_DIR / "BX-Books.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
users = pd.read_csv(DATA_DIR / "BX-Users.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
ratings = pd.read_csv(DATA_DIR / "BX-Book-Ratings.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
books.columns = ["isbn", "title", "author", "year", "publisher", "image_s", "image_m", "image_l"]
users.columns = ["user_id", "location", "age"]
ratings.columns = ["user_id", "isbn", "rating"]
data = ratings.merge(books[["isbn", "title", "author", "image_m"]], on="isbn", how="inner")
data = data[data["rating"] > 0].copy()
print(f"Explicit ratings: {len(data):,} | readers: {data['user_id'].nunique():,} | books: {data['title'].nunique():,}")
display(data.head())


Explicit ratings: 383,842 | readers: 68,091 | books: 135,567


,user_id,isbn,rating,title,author,image_m
1,276726,0155061224,5,Rites of Passage,Judith Rae,http://images.amazon.com/images/P/0155061224.0...
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,http://images.amazon.com/images/P/052165615X.0...
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,http://images.amazon.com/images/P/0521795028.0...
6,276744,038550120X,7,A Painted House,JOHN GRISHAM,http://images.amazon.com/images/P/038550120X.0...
13,276747,0060517794,9,Little Altars Everywhere,Rebecca Wells,http://images.amazon.com/images/P/0060517794.0...


## 3. Control sparsity with transparent support rules


In [3]:
reader_activity = data.groupby("user_id").size()
book_activity = data.groupby("title").size()
active_readers = reader_activity[reader_activity >= 35].index
popular_books = book_activity[book_activity >= 25].index
filtered = data[data["user_id"].isin(active_readers) & data["title"].isin(popular_books)].copy()
pivot = filtered.pivot_table(index="title", columns="user_id", values="rating", fill_value=0)
print(f"Filtered interactions: {len(filtered):,} | matrix: {pivot.shape[0]} books × {pivot.shape[1]} readers")
display(book_activity.sort_values(ascending=False).head(10).rename("rating_count").to_frame())


Filtered interactions: 38,276 | matrix: 1719 books × 1776 readers


,rating_count
title,
The Lovely Bones: A Novel,707
Wild Animus,581
The Da Vinci Code,494
The Secret Life of Bees,406
The Nanny Diaries: A Novel,393
The Red Tent (Bestselling Backlist),383
Bridget Jones's Diary,377
A Painted House,366
Life of Pi,336


## 4. Fit item-based nearest neighbours


In [4]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=6)
model.fit(pivot.values)

def recommend_book(title, top_n=5):
    if title not in pivot.index:
        raise KeyError(f"{title!r} is not available after support filtering")
    position = pivot.index.get_loc(title)
    distances, indices = model.kneighbors(pivot.iloc[position].to_numpy().reshape(1, -1), n_neighbors=top_n + 1)
    result = pd.DataFrame({"title": pivot.index[indices[0][1:]], "cosine_similarity": 1 - distances[0][1:]})
    return result

query_title = "Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))" if "Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))" in pivot.index else pivot.index[0]
print(f"Recommendations for: {query_title}")
display(recommend_book(query_title).round(4))


Recommendations for: Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))


,title,cosine_similarity
0,Harry Potter and the Chamber of Secrets (Book 2),0.3880
1,Harry Potter and the Prisoner of Azkaban (Book 3),0.3344
2,Harry Potter and the Goblet of Fire (Book 4),0.3149
3,Harry Potter and the Order of the Phoenix (Boo...,0.2289
4,Like Water for Chocolate: A Novel in Monthly I...,0.1712


## 5. Coverage and sample recommendation checks


In [5]:
queries = pivot.index[[0, len(pivot) // 3, 2 * len(pivot) // 3]].tolist()
for title in queries:
    print(f"\nTop neighbours for {title}")
    display(recommend_book(title, top_n=4).round(4))
print(f"Catalog coverage after support filters: {len(pivot) / data['title'].nunique():.2%}")



Top neighbours for 1984

Top neighbours for Holidays on Ice : Stories

Top neighbours for The Amber Spyglass (His Dark Materials, Book 3)
Catalog coverage after support filters: 1.27%


,title,cosine_similarity
0,Animal Farm,0.2918
1,Brave New World,0.2518
2,The Handmaid's Tale,0.1995
3,The Catcher in the Rye,0.1869


,title,cosine_similarity
0,The Talented Mr. Ripley (Vintage Crime/Black L...,0.3060
1,Catch 22,0.2135
2,Cat's Eye,0.1919
3,The Robber Bride,0.1873


,title,cosine_similarity
0,"The Subtle Knife (His Dark Materials, Book 2)",0.6565
1,"The Golden Compass (His Dark Materials, Book 1)",0.4608
2,Speaker for the Dead (Ender Wiggins Saga (Pape...,0.2055
3,Northern Lights (His Dark Materials S.),0.2037


## 6. Findings and limitations

- Explicit-rating support rules make the similarity matrix more stable but reduce catalog coverage.
- The method cannot recommend new books with no interactions.
- Similarity is not a claim of reader satisfaction.
- Production evaluation should include time-aware precision@k, recall@k, diversity, novelty, and online feedback.
